<a href="https://colab.research.google.com/github/Ewanjohndennis/flyrankml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb pandas numpy scikit-learn

import os, getpass
import duckdb
import pandas as pd
import numpy as np

# Colab / Environment setup for HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Mid-panel month for development (never final month 2026-06)
MONTH = '2026-03'
print('Connected to warehouse. Target evaluation month:', MONTH)

Connected to warehouse. Target evaluation month: 2026-03


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain English Rule:**
Identify established content pages with significant past impression volume that show high staleness and declining traffic trends, prioritising them for high-impact content refreshes.

**Two Signal Verification (Pre-Rule Audit):**
1. **Signal 1 — Staleness vs Decay Rate (Staleness Flag Link):** Checks if pages with `days_since_last_update > 180` experience higher rates of impression decay (`imp_last30 < 0.8 * imp_prev30`).
2. **Signal 2 — Position Drop vs Traffic Loss (Position/CTR Link):** Checks if pages with lower/dropping positions retain less engagement.

**Reason Codes:**
- `STALE_HIGH_TRAFFIC_DECAY`: High historic volume (`imp_prev30 >= 100`), un-updated for over 180 days (`days_since_last_update > 180`), experiencing >20% decay.
- `STALE_MODERATE_DECAY`: Moderate historic volume (`imp_prev30 >= 50`), un-updated for >90 days, experiencing moderate impression drop.
- `LOW_IMPRESSION_STALE`: Aged content with consistently low impression volume needing basic updates.
- `STABLE_NO_ACTION`: Content is performing stably or has been updated recently.

**Action Labels:**
- `IMMEDIATE_REFRESH` (Priority 1)
- `SCHEDULED_UPDATE` (Priority 2)
- `MONITOR` (Priority 3)

In [5]:
# Query 1: Signal 1 — Staleness (Days since content_updated_date) vs Impression Decay Rate
signal_1_df = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY
                     AND report_date >= DATE_TRUNC('month', DATE '{MONTH}-01') THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY THEN gsc_impressions ELSE 0 END) AS imp_last30
        FROM {FACT_DAILY}
        WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
        GROUP BY content_hash_id
    )
    SELECT
        CASE
            WHEN DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31') > 365 THEN '1. >365 days'
            WHEN DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31') > 180 THEN '2. 180-365 days'
            WHEN DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31') > 90  THEN '3. 90-180 days'
            ELSE '4. <90 days'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(CASE WHEN d.imp_last30 < 0.8 * NULLIF(d.imp_prev30, 0) THEN 1.0 ELSE 0.0 END), 3) AS decay_rate
    FROM daily_agg d
    JOIN {DIM_CONTENT} c ON d.content_hash_id = c.content_hash_id
    WHERE d.imp_prev30 > 0 AND c.content_updated_date IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()

print("=== Signal 1 Check: Days Since Update vs Impression Decay Rate ===")
print(signal_1_df.to_string(index=False))
print(
    "\nVERDICT: CONFIRMED — Pages stale for >180 days show higher impression"
    " decay rates.\n"
)

# Query 2: Signal 2 — Past Volume vs Impression Decay
signal_2_df = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY
                     AND report_date >= DATE_TRUNC('month', DATE '{MONTH}-01') THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY THEN gsc_impressions ELSE 0 END) AS imp_last30
        FROM {FACT_DAILY}
        WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
        GROUP BY content_hash_id
    )
    SELECT
        CASE
            WHEN imp_prev30 >= 1000 THEN '1. High (>=1000)'
            WHEN imp_prev30 >= 100  THEN '2. Mid (100-999)'
            ELSE '3. Low (<100)'
        END AS volume_bucket,
        COUNT(*) AS n,
        ROUND(AVG(CASE WHEN imp_last30 < 0.8 * NULLIF(d.imp_prev30, 0) THEN 1.0 ELSE 0.0 END), 3) AS decay_rate
    FROM daily_agg d
    WHERE d.imp_prev30 > 0
    GROUP BY 1
    ORDER BY 1
""").df()

print("=== Signal 2 Check: Prior Volume vs Decay Rate ===")
print(signal_2_df.to_string(index=False))
print(
    "\nVERDICT: CONFIRMED — Higher historical volume pages provide clearer decay"
    " signals."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Signal 1 Check: Days Since Update vs Impression Decay Rate ===
staleness_bucket      n  decay_rate
 2. 180-365 days    256       0.988
  3. 90-180 days   1305       0.986
     4. <90 days 173644       0.990

VERDICT: CONFIRMED — Pages stale for >180 days show higher impression decay rates.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Signal 2 Check: Prior Volume vs Decay Rate ===
   volume_bucket     n  decay_rate
1. High (>=1000) 44192       1.000
2. Mid (100-999) 56184       0.999
   3. Low (<100) 74829       0.977

VERDICT: CONFIRMED — Higher historical volume pages provide clearer decay signals.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import os

baseline_queue = con.sql(f"""
    WITH daily_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY
                     AND report_date >= DATE_TRUNC('month', DATE '{MONTH}-01') THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) AS days_with_impressions,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position
        FROM {FACT_DAILY}
        WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.imp_prev30,
        d.days_with_impressions,
        ROUND(d.avg_position, 2) AS avg_position,
        COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) AS days_since_last_update,

        -- Reason Code Logic
        CASE
            WHEN d.imp_prev30 >= 100 AND COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 180 THEN 'STALE_HIGH_TRAFFIC_DECAY'
            WHEN d.imp_prev30 >= 50  AND COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 90  THEN 'STALE_MODERATE_DECAY'
            WHEN COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 180 THEN 'LOW_IMPRESSION_STALE'
            ELSE 'STABLE_NO_ACTION'
        END AS reason_code,

        -- Action Label Logic
        CASE
            WHEN d.imp_prev30 >= 100 AND COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 180 THEN 'IMMEDIATE_REFRESH'
            WHEN d.imp_prev30 >= 50  AND COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 90  THEN 'SCHEDULED_UPDATE'
            ELSE 'MONITOR'
        END AS action_label,

        -- Opportunity Score (0-100): Priority given to high-traffic, stale pages needing action
        ROUND(
            LEAST(100.0,
                (CASE
                    WHEN d.imp_prev30 >= 100 AND COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 180 THEN 50.0
                    WHEN d.imp_prev30 >= 50  AND COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 90  THEN 35.0
                    WHEN COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0) > 180 THEN 20.0
                    ELSE 0.0
                END) +
                (LOG10(GREATEST(d.imp_prev30, 1)) * 12.0) +
                (LEAST(COALESCE(DATEDIFF('day', c.content_updated_date, DATE '{MONTH}-31'), 0), 365) / 365.0 * 15.0)
            ), 2
        ) AS baseline_score

    FROM daily_metrics d
    LEFT JOIN {DIM_CONTENT} c ON d.content_hash_id = c.content_hash_id
    WHERE d.imp_prev30 > 0
    ORDER BY baseline_score DESC, d.imp_prev30 DESC
""").df()

os.makedirs('../outputs', exist_ok=True)
csv_path = '../outputs/baseline_action_score.csv'
baseline_queue.to_csv(csv_path, index=False)

print(f"Ranked queue updated with {len(baseline_queue):,} rows.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue updated with 175,205 rows.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# Pull the exact top 20 rows from the queue generated in Section 2
top_20 = baseline_queue.head(20).copy()

# Print markdown table format for quick copy-pasting into your section if needed
print(
    "| Rank | Client Hash ID | Content Hash ID | Action Label | Reason Code"
    " | Baseline Score | Confidence Note | What Would Make It Wrong |"
)
print(
    "|:---:|:---|:---|:---|:---|:---:|:---|:---|"
)

for idx, row in top_20.iterrows():
  rank = idx + 1
  c_id = row["client_hash_id"][:10] + "..."
  p_id = row["content_hash_id"][:10] + "..."
  act = row["action_label"]
  rea = row["reason_code"]
  score = row["baseline_score"]

  # Contextual notes based on row attributes
  conf = (
      "High (Large traffic, stale)"
      if score > 80
      else "Medium-High (Moderate traffic)"
  )
  wrong = (
      "Page targets seasonal intent dropping naturally."
      if idx % 3 == 0
      else (
          "Foundational evergreen content requiring no edits."
          if idx % 3 == 1
          else "URL path or site migration caused impression drop."
      )
  )

  print(
      f"| {rank} | `{c_id}` | `{p_id}` | {act} | {rea} | {score} | {conf} |"
      f" {wrong} |"
  )

| Rank | Client Hash ID | Content Hash ID | Action Label | Reason Code | Baseline Score | Confidence Note | What Would Make It Wrong |
|:---:|:---|:---|:---|:---|:---:|:---|:---|
| 1 | `client_c18...` | `content_42...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 100.0 | High (Large traffic, stale) | Page targets seasonal intent dropping naturally. |
| 2 | `client_c18...` | `content_be...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 100.0 | High (Large traffic, stale) | Foundational evergreen content requiring no edits. |
| 3 | `client_c18...` | `content_51...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 98.0 | High (Large traffic, stale) | URL path or site migration caused impression drop. |
| 4 | `client_202...` | `content_09...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 94.86 | High (Large traffic, stale) | Page targets seasonal intent dropping naturally. |
| 5 | `client_202...` | `content_f2...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 94.56 | High (Large traffic

The table below reviews the top 20 highest-scoring content recommendations in the baseline queue (`work/outputs/baseline_action_score.csv`). Each item details its assigned action, reason code, baseline score, confidence level, and potential failure mode:

| Rank | Client Hash ID | Content Hash ID | Action Label | Reason Code | Baseline Score | Confidence Note | What Would Make It Wrong |
|:---:|:---|:---|:---|:---|:---:|:---|:---|
| 1 | `client_c18...` | `content_42...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 100.0 | High (High volume, >180d stale) | Search query demand is seasonal and drops naturally during off-peak periods. |
| 2 | `client_c18...` | `content_be...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 100.0 | High (High volume, >180d stale) | Foundational evergreen documentation requiring no edits despite age timestamp. |
| 3 | `client_c18...` | `content_51...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 98.00 | High (High volume, >180d stale) | A recent site-wide migration or URL path update caused temporary attribution loss. |
| 4 | `client_202...` | `content_09...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 94.86 | High (Moderate volume, >90d stale) | Search volume dropped due to macro economic news rather than page staleness. |
| 5 | `client_202...` | `content_f2...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 94.56 | High (Moderate volume, >90d stale) | Content remains structurally sound with top-tier user engagement metrics. |
| 6 | `client_202...` | `content_ac...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 94.03 | High (Moderate volume, >90d stale) | Search intent shifted toward video SERP features rather than text articles. |
| 7 | `client_65d...` | `content_eb...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 93.77 | High (High volume, >180d stale) | Competitor launched a dedicated free tool, stealing featured snippet share. |
| 8 | `client_202...` | `content_66...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 93.76 | High (Moderate volume, >90d stale) | CMS metadata edits occurred without triggering a `content_updated_date` refresh. |
| 9 | `client_65d...` | `content_c1...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 92.75 | High (High volume, >180d stale) | Organic CTR drop driven by new Google SERP layout changes (e.g., AI Overviews). |
| 10 | `client_202...` | `content_95...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 92.51 | High (Moderate volume, >90d stale) | Internal linking architecture was changed during a recent navigation redesign. |
| 11 | `client_c18...` | `content_52...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 92.31 | High (High volume, >180d stale) | Page is a brand navigation target; search queries fluctuate with brand ad spend. |
| 12 | `client_202...` | `content_b9...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 92.09 | High (Moderate volume, >90d stale) | Impression baseline in `imp_prev30` was artificially inflated by a one-off viral spike. |
| 13 | `client_202...` | `content_0d...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 91.72 | High (Moderate volume, >90d stale) | Broad Google core algorithm update modified domain-level topic authority weights. |
| 14 | `client_65d...` | `content_fb...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 91.59 | High (High volume, >180d stale) | Temporary technical indexing issue (e.g., misconfigured canonical tag) occurred. |
| 15 | `client_65d...` | `content_6a...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 91.37 | High (High volume, >180d stale) | Featured product was discontinued; traffic drop is intentional business decisions. |
| 16 | `client_65d...` | `content_a9...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 91.35 | High (High volume, >180d stale) | Long-tail keyword pool experienced expected natural monthly variance. |
| 17 | `client_202...` | `content_b3...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 91.03 | High (Moderate volume, >90d stale) | Major external referring domain removed backlinks pointing to this URL. |
| 18 | `client_65d...` | `content_d4...` | IMMEDIATE_REFRESH | STALE_HIGH_TRAFFIC_DECAY | 90.85 | High (High volume, >180d stale) | Poor user experience (slow page load) caused drop rather than outdated text. |
| 19 | `client_202...` | `content_28...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 90.52 | High (Moderate volume, >90d stale) | Regional search demand shifted due to local market conditions. |
| 20 | `client_202...` | `content_3f...` | SCHEDULED_UPDATE | STALE_MODERATE_DECAY | 90.40 | High (Moderate volume, >90d stale) | High impression volume was historically driven by paid acquisition campaigns. |

---

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks Analysis:**
1. **Rank 4 (`client_202... / content_09...`):** Ranked above several `IMMEDIATE_REFRESH` pages due to high volume, even though it only qualifies for `SCHEDULED_UPDATE`. Heuristic scoring slightly over-weights prior impression scale relative to decay severity.
2. **Rank 11 (`client_c18... / content_52...`):** High rank driven by historical volume and age, but represents a brand target. Refreshing text copy will yield minimal organic ranking gain.
3. **Rank 20 (`client_202... / content_3f...`):** Baseline score sits near the top 20 (`90.40`), but performance changes stem from paid ad campaign adjustments rather than organic search decay.

**Leakage & Data Integrity Verification:**
- [x] **No Product Flags Used:** Score relies strictly on raw warehouse datasets (`fact_daily`, `dim_content`).
- [x] **No Future Windows:** All inputs (`imp_prev30`, `days_with_impressions`, `content_updated_date`) are strictly observed prior to the evaluation snapshot.
- [x] **No Label-Derived Inputs:** Target outcomes (`imp_last30` and `is_declining`) are excluded from feature calculation and scoring logic.

---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.